## **Imports**

In [1]:
from src.millionaire_client import AuthenticationError, MillionaireClient
from src.benchmark import Benchmark
from dotenv import load_dotenv
import os
from src.models import ExperimentConfig,ApproachType
from Marcelo.src.guesser.guesser import MarceloGuesser

## **Auth**

In [2]:
load_dotenv(dotenv_path=os.path.join("Marcelo", ".env"))


True

In [3]:
API_URL = "http://131.175.15.22:51111/"

USERNAME = os.getenv("MILLIONAIRE_USERNAME")
PASSWORD = os.getenv("MILLIONAIRE_PASSWORD")


In [4]:
client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


Welcome, MTKY! (Role: student)

=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)


In [ ]:
from Marcelo.src.guesser.configs import INFERENCE_MODEL,EMBEDDING_MODEL

marcelo_experiment_config = ExperimentConfig(
experiment_id="1",
username="Marcelo",
notes="first_test",
approach=ApproachType.RAG,
inference_model=INFERENCE_MODEL,
inference_model_size=2,
embedding_model=EMBEDDING_MODEL,
embedding_model_size=0.2,
is_rag=True
)

guesser = MarceloGuesser(marcelo_experiment_config,embedding_model_name=EMBEDDING_MODEL,inference_model_name=INFERENCE_MODEL)

In [ ]:
benchmark = Benchmark(marcelo_experiment_config,guesser,client)
benchmark.run(3)

Started game in competition: Entertainment

--- Level 1 ---
Q: In which year was 'Spider-Man 3' released?
  [0] 2010
  [1] 2007
  [2] 2004
  [3] 2002

Guesser is thinking...
Guesser chose option: 3 (Time: 21.86s)
CONGRATULATIONS! Final earnings: $0.00
Started game in competition: Entertainment

--- Level 1 ---
Q: What is the primary reason Taylor became a prominent figure in HIV/AIDS activism?
  [0] Professional obligation
  [1] Personal experience with the disease
  [2] Public image and media attention
  [3] Pressure from the government

Guesser is thinking...


KeyboardInterrupt: 

## **Model Definition**

In [7]:
MODEL_ID = "google/gemma-4-e2b-it"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=TOKEN,
    device_map="cpu"
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

## **Quiz**

In [ ]:
# Prompt

system_instructions = {"role": "system", 
    "content":
    """
    You are an expert data extraction and Q&A assistant. Your task is to read the provided context and answer the multiple-choice question. 
    CRITICAL RULE: You must reply ONLY with the exact letter of the correct option (A, B, C, or D). Do not provide any explanations, introductory phrases, or punctuation other than the single uppercase letter.
    Question: Which of the following best describes Sicily?
    A) A small lake in northern Europe
    B) A landlocked country in Asia
    C) The largest island in the Mediterranean Sea
    D) A mountain range in France
    Answer: C

    Question: {user_question}
    A) {option_a}
    B) {option_b}
    C) {option_c}
    D) {option_d}
    Answer:
    """
}
messages = [
    system_instructions,
    {"role": "user", "content": "Write a short joke about saving RAM."},
]

# Process input
text = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True, 
    enable_thinking=False
)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=1024)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
processor.parse_response(response)


{'role': 'assistant', 'content': 'B'}

In [ ]:
from Marcelo.src.guesser.ingestion.pipelines.zim_ingestion import IngestionPipeline
from src.guesser.configs import EMBEDDING_MODEL

In [2]:
RAW_DATA_FILE = "src/guesser/data/raw/wikipedia_en_simple_all_maxi_2026-02.zim"
DB_PATH = "src/guesser/context_db/"
COLLECTION_NAME = "wikipedia_test"
pipeline = IngestionPipeline(RAW_DATA_FILE,EMBEDDING_MODEL,DB_PATH,COLLECTION_NAME)



In [7]:
pipeline.process(50)


Starting extraction and chunking...
TextNode(id_='9c091043-7c78-455a-b6fc-d7271b62c26f', embedding=None, metadata={'title': '!', 'path': '!'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='4ee4b533-64b7-48d0-8087-b7aba144b82e', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'title': '!', 'path': '!'}, hash='7bb5b0b46ef7993f1ccb3be09980b9376eefe1f75d8ed02c70470f35795e3034'), <NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='32282ffb-4238-4276-9bca-c401b14be222', node_type=<ObjectType.TEXT: '1'>, metadata={}, hash='ce07ac628f54940dd4338bb20143a3f760c54cbf10d9ce28d6d520213b433130')}, metadata_template='{key}: {value}', metadata_separator='\n', text='Exclamation mark "!" redirects here. For other uses, see ! (disambiguation) . Not to be confused with I . An exclamation mark. An exclamation mark ( ! ) is a punctuation mark. It is used to show strong emotion at the end of a sentence or after an in

In [4]:
from src.guesser.ingestion.loader import Loader
from src.guesser.engine.guesser_engine import GuesserEngine
from src.guesser.configs import INFERENCE_MODEL

In [8]:
print("Conectando ao banco de dados ChromaDB...")

loader = Loader(DB_PATH)
index = loader.get_index()

print(f"Inicializando o Llama 3.2 via Ollama...")
engine = GuesserEngine(index,INFERENCE_MODEL)

Conectando ao banco de dados ChromaDB...
Inicializando o Llama 3.2 via Ollama...


In [10]:
pergunta_teste = (
    "Question: What is the name of the Minecraft main character?\n"
    "[0] Steve\n"
    "[1] Mario \n"
    "[2] Luigi\n"
    "[3] Bowser"
)
engine.answer_question(pergunta_teste)

{'answer': 'Empty Response', 'sources': []}

In [8]:

from datasets import load_dataset
from llama_index.core.schema import TextNode
import hashlib

In [11]:
dataset_ciencias = load_dataset("sciq", split="train")

In [12]:
len(dataset_ciencias)

11679

In [ ]:
nodes = []
for item in dataset_ciencias.select(range(3)):
    contexto = item['support']
    print(item["question"])
    print("\n")
    node_id = hashlib.md5(contexto.encode('utf-8')).hexdigest()
    
    node = TextNode(
        id_=node_id,
        text=contexto,
        metadata={
            "pergunta": item['question'],
            "resposta": item['correct_answer']
        }
    )
    nodes.append(node)

What type of organism is commonly used in preparation of foods such as cheese and yogurt?


What phenomenon makes global winds blow northeast to southwest or the reverse in the northern hemisphere and northwest to southeast or the reverse in the southern hemisphere?


Changes from a less-ordered state to a more-ordered state (such as a liquid to a solid) are always what?




In [10]:
nodes[1]

TextNode(id_='c8bb18139eade1c3c7fc23beaa3be3c8', embedding=None, metadata={'pergunta': 'What phenomenon makes global winds blow northeast to southwest or the reverse in the northern hemisphere and northwest to southeast or the reverse in the southern hemisphere?', 'resposta': 'coriolis effect'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Without Coriolis Effect the global winds would blow north to south or south to north. But Coriolis makes them blow northeast to southwest or the reverse in the Northern Hemisphere. The winds blow northwest to southeast or the reverse in the southern hemisphere.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}')

In [1]:
from src.guesser.ingestion.extractors.dataset_extractor import DatasetExtractor
from src.guesser.ingestion.extractors.configs import HF_SCIENCE_DATASET,HF_MATH_DATASET

In [5]:
science_extractor = DatasetExtractor(HF_SCIENCE_DATASET)
science_nodes = science_extractor.extract(100)

In [4]:
math_extractor = DatasetExtractor(HF_MATH_DATASET)
math_nodes = math_extractor.extract(100)

In [9]:
math_nodes

[TextNode(id_='984523cca1526ec2b87f6479886ad495', embedding=None, metadata={'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}'),
 TextNode(id_='1e5b09740d26a80302474381612cb98d', embedding=None, metadata={'answer': 'Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.\nWorking 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.\n#### 10'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Weng ea

In [1]:
DB_PATH = "src/guesser/context_db/"
COLLECTION_NAME = "Science_Nature"

In [2]:
from src.guesser.ingestion.pipelines.dataset_ingestion import DatasetIngestionPipeline
from src.guesser.ingestion.extractors.configs import HF_SCIENCE_DATASET,HF_MATH_DATASET
from src.guesser.configs import EMBEDDING_MODEL

In [3]:
pipeline = DatasetIngestionPipeline(HF_MATH_DATASET,EMBEDDING_MODEL,DB_PATH,COLLECTION_NAME)

In [4]:
pipeline.process(50)

Starting extraction and chunking...


TextNode(id_='dba7a176-23e0-496f-82ae-ad12d5fab03d', embedding=None, metadata={'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='fe069436-a372-42e1-852b-fa90c1b90128', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}, hash='7fdfea1875dd9b27a3ec5298b12c8bddbfd1cf61c0caaaa64a8d6398d4efb77a')}, metadata_template='{key}: {value}', metadata_separator='\n', text='Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', mimetype='text/plain', start_char_idx=0, end_char_idx=155, text_template='{metadata_str}\n\n{content}')
TextNode

In [1]:
from src.guesser.ingestion.loader import Loader
from src.guesser.engine.guesser_engine import GuesserEngine
from src.guesser.configs import INFERENCE_MODEL
from src.guesser.configs import EMBEDDING_MODEL
from src.guesser.engine.prompts import MCQ_PROMPT_MATHS
print("Conectando ao banco de dados ChromaDB...")
DB_PATH = "src/guesser/context_db/"
COLLECTION_NAME = "Science_Nature"
loader = Loader(DB_PATH,EMBEDDING_MODEL)
index = loader.get_index(COLLECTION_NAME)

engine = GuesserEngine(index,INFERENCE_MODEL,top_k=3,prompt=MCQ_PROMPT_MATHS)


Conectando ao banco de dados ChromaDB...


In [2]:
# BASEADA EM: "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?"
# LÓGICA: (Custo Total - Metade) - Doação 1 - (Doação 1 * 2) = Restante
# Resposta correta: [1]
pergunta_dinamica_1 = (
    "Question: Lucas is saving money for a new skateboard which costs $120. Lucas has only half of the money he needs. His sister gave him $15 to help, and his uncle gave him twice as much as his sister. How much more money does Lucas need to buy the skateboard?\n"
    "[0] $10\n"
    "[1] $15\n"
    "[2] $30\n"
    "[3] $60"
)
engine.answer_question(pergunta_dinamica_1)

{'answer': "Lucas initially had half of the cost, which is $120 / 2 = $<<120/2=60>>60. His sister gave him $15 and his uncle twice that amount ($15 x 2), so he received an additional $30 from them combined. This means Lucas has a total of $60 + $15 + $30 = $<<60+15+30=105>>105 now. To find out how much more money he needs, subtract this amount from the skateboard's cost: $120 - $105 = $<<120-105=15>>15\n#### 15",
 'sources': ['Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?',
  'Tobias is buying a new pair of shoes that costs $95. He has been saving up his money each month for the past three months. ',
  'Betty is saving money for a new wallet which costs $100. '],
 'resolution_method': ["In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.\nBetty's grandparents gave her 15 * 2 = $<<15*2=30>>30.\nThis means, Betty needs 100 -

In [4]:
# BASEADA EM: "Randy has 60 mango trees on his farm. He also has 5 less than half as many coconut trees as mango trees. How many trees does Randy have in all on his farm?"
# LÓGICA: Árvore A + ((Árvore A / 2) - Subtração) = Total
# Resposta correta: [2]
pergunta_dinamica_4 = (
    "Question: Farmer Joe has 80 apple trees in his orchard. He also has 10 less than half as many pear trees as apple trees. How many trees does Joe have in total?\n"
    "[0] 90 trees\n"
    "[1] 100 trees\n"
    "[2] 110 trees\n"
    "[3] 120 trees"
)
engine.answer_question(pergunta_dinamica_4)

{'answer': "Half of the number of Farmer Joe's apple trees is 80/2 = <<80/2=40>>40 trees.\nSo, he has 40 - 10 = <<40-10=30>>30 pear trees.\nTherefore, in total, Farmer Joe has 80 + 30 = <<80+30=110>>110 trees on his orchard.\n#### 110",
 'sources': ['He also has 5 less than half as many coconut trees as mango trees. How many trees does Randy have in all on his farm?',
  'Randy has 60 mango trees on his farm. ',
  'There are only 25% as many green flowers as there are yellow and purple flowers. How many flowers does Mark have in his garden?'],
 'resolution_method': ["Half of the number of Randy's mango trees is 60/2 = <<60/2=30>>30 trees.\nSo Randy has 30 - 5 = <<30-5=25>>25 coconut trees.\nTherefore, Randy has 60 + 25 = <<60+25=85>>85 treeson his farm.\n#### 85",
  "Half of the number of Randy's mango trees is 60/2 = <<60/2=30>>30 trees.\nSo Randy has 30 - 5 = <<30-5=25>>25 coconut trees.\nTherefore, Randy has 60 + 25 = <<60+25=85>>85 treeson his farm.\n#### 85",
  "There are 80/100 * 

In [5]:
pergunta_teste = (
    "Question: Julie is reading a 120-page book. Yesterday, she read 12 pages, and today, she read twice as many pages as yesterday. If she plans to read exactly half of the remaining pages tomorrow, how many pages should she read?\n"
    "[0] 36 pages\n"
    "[1] 42 pages\n"
    "[2] 48 pages\n"
    "[3] 84 pages"
)
engine.answer_question(pergunta_teste)

{'answer': 'Julie has already read a total of 12 (yesterday) + 2 * 12 (today) = <<12+2*12=36>>36 pages. This leaves her with 120 - 36 = <<120-36=84>>84 pages remaining in the book. If she plans to read half of these leftover pages tomorrow, then she should aim for reading 84/2 = <<84/2=42>>42 pages.\n#### 42',
 'sources': ['Julie is reading a 120-page book. ',
  'Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?',
  'Joy can read 8 pages of a book in 20 minutes. How many hours will it take her to read 120 pages?'],
 'resolution_method': ['Maila read 12 x 2 = <<12*2=24>>24 pages today.\nSo she was able to read a total of 12 + 24 = <<12+24=36>>36 pages since yesterday.\nThere are 120 - 36 = <<120-36=84>>84 pages left to be read.\nSince she wants to read half of the remaining pages tomorrow, then she should read 84/2 = <<84/2=42>>42 pages.\n#### 42',
  'M

In [5]:
# Resposta correta: [0]
pergunta_teste_1 = (
    "Question: Marco needs money for a new pijama which costs $50. Marco has only half of the money he needs. His friends decided to give him $5 for that purpose, and his grandparents twice as much as his friends. How much more money does Marco need to buy the pijama?\n"
    "[0] $5\n"
    "[1] $10\n"
    "[2] $100\n"
    "[3] $50"
)
engine.answer_question(pergunta_teste_1)


{'answer': "Marco has half of the amount needed for the pajamas, which is $50 / 2 = $<<50/2=25>>25. His friends contribute with $5 and his grandparents give twice what his friends did, so that's an additional $5 * 2 = $<<5*2=10>>10 from them. Marco still needs a total of $25 - ($5 + $10) = $<<25-5-10=10>>10 more to buy the pajamas.\n#### 10",
 'sources': ['Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?',
  'Rachel and Sara want to attend a beauty and modeling contest. They both want to buy new pairs of shoes and dresses. ',
  'Betty is saving money for a new wallet which costs $100. '],
 'resolution_method': ["In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.\nBetty's grandparents gave her 15 * 2 = $<<15*2=30>>30.\nThis means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.\n#### 5",
  'The cost Rachel should 

In [4]:
# Resposta correta: [1]
pergunta_teste_2 = (
    "Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?\n"
    "[0] $8\n"
    "[1] $10\n"
    "[2] $12\n"
    "[3] $15"
)
engine.answer_question(pergunta_teste_2)


{'answer': '1',
 'sources': ['Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. ',
  'How much did she earn?',
  'Tina makes $18.00 an hour.  ',
  'If she works more than 8 hours per shift, she is eligible for overtime, which is paid by your hourly wage + 1/2 your hourly wage.  If she works 10 hours every day for 5 days, how much money does she make?',
  'Lisa, Jack, and Tommy earned $60 from washing cars all week. However, half of the $60 was earned by Lisa. ']}

In [5]:
# Resposta correta: [2]
pergunta_teste_3 = (
    "Question: James writes a 3-page letter to 2 different friends twice a week. How many pages does he write a year?\n"
    "[0] 312 pages\n"
    "[1] 500 pages\n"
    "[2] 624 pages\n"
    "[3] 700 pages"
)
engine.answer_question(pergunta_teste_3)


{'answer': '2',
 'sources': ['James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?',
  'Julie is reading a 120-page book. ',
  'Roque walks to and from work three times a week and rides his bike to and from work twice a week. How many hours in total does he take to get to and from work a week with walking and biking?',
  'Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?',
  'Joy can read 8 pages of a book in 20 minutes. How many hours will it take her to read 120 pages?']}

In [6]:
# Resposta correta: [3]
pergunta_teste_4 = (
    "Question: Albert buys 2 large pizzas and 2 small pizzas. A large pizza has 16 slices and a small pizza has 8 slices. If he eats it all, how many pieces does he eat that day?\n"
    "[0] 24 pieces\n"
    "[1] 32 pieces\n"
    "[2] 40 pieces\n"
    "[3] 48 pieces"
)
engine.answer_question(pergunta_teste_4)


{'answer': '3',
 'sources': ['Albert is wondering how much pizza he can eat in one day. He buys 2 large pizzas and 2 small pizzas. ',
  'A large pizza has 16 slices and a small pizza has 8 slices. If he eats it all, how many pieces does he eat that day?',
  'Ann, Bill, Cate, and Dale each buy personal pan pizzas cut into 4 pieces. If Bill and Dale eat 50% of their pizzas and Ann and Cate eat 75% of the pizzas, how many pizza pieces are left uneaten?',
  'To make pizza, together with other ingredients, Kimber needs 10 cups of water, 16 cups of flour, and 1/2 times as many teaspoons of salt as the number of cups of flour. Calculate the combined total number of cups of water, flour, and teaspoons of salt that she needs to make the pizza.',
  'Her brother Billy goes trick-or-tricking in a neighboring subdivision where he gets 11 pieces of candy per house. If the first subdivision has 60 houses and the second subdivision has 75 houses, how many more pieces of candy does Anna get?']}

In [8]:
# Resposta correta: [2]
pergunta_teste_5 = (
    "Question: Mark has a garden with flowers. Ten of them are yellow, and there are 80% more of those in purple. How many purple flowers does he have?\n"
    "[0] 8 flowers\n"
    "[1] 12 flowers\n"
    "[2] 18 flowers\n"
    "[3] 80 flowers"
)
engine.answer_question(pergunta_teste_5)

{'answer': '2',
 'sources': ['Mark has a garden with flowers. He planted plants of three different colors in it. Ten of them are yellow, and there are 80% more of those in purple. ',
  'There are only 25% as many green flowers as there are yellow and purple flowers. How many flowers does Mark have in his garden?',
  'There are 5 houses on a street, and each of the first four houses has 3 gnomes in the garden. If there are a total of 20 gnomes on the street, how many gnomes does the fifth house have?',
  'He also has 5 less than half as many coconut trees as mango trees. How many trees does Randy have in all on his farm?',
  'Randy has 60 mango trees on his farm. ']}